In [38]:
import os 
import datetime
import pandas as pd
from time import sleep
from pydantic_ai import Agent
from dotenv import load_dotenv
from pydantic_ai.models.google import GoogleModel
from pydantic_ai.providers.google import GoogleProvider

In [39]:
load_dotenv(override=True)
API_KEY = os.environ['PAID_GEMMA_API_KEY']

In [40]:
SYSTEM_PROMPT = """
    You are a helpful assistant that converts the url link of a news article to sentence like topic. As an input you will get bulk of links as string.
    Your task is to return two or maximum three sentences that are going to give some type of description about what happened to gold market or gold prices.
    While generating the answer make sure that:
    -Try to approximate the situation as close as you can.
    -Answer only the topic, do not add extra discussion. 
    -Do not format the answer.  
    Make sure that you capture all the necessary information and the details about the text that is provided you.
"""

provider = GoogleProvider(api_key=API_KEY)
model = GoogleModel("gemini-2.0-flash", provider=provider)
agent = Agent(model=model, system_prompt=SYSTEM_PROMPT, output_type=str)

In [41]:
base_path = r"C:\Users\User\dev\training_model\gdelt_data_fetch\gdelt_gold_filtered"

In [43]:
start_date = datetime.date(2014, 6, 5)
end_date = datetime.date(2025, 8, 10)

results = []
missing_dates = []
current_date = start_date

while current_date <= end_date:
    print(f"Processing for date: {current_date}")

    date_str = current_date.strftime("%Y%m%d")
    file_name = f"{date_str}_gold_filtered.csv"
    full_path = os.path.join(base_path, file_name)

    try:
        df = pd.read_csv(full_path)

        url_bulk = ""
        for url in df['SOURCEURL']:
            url_bulk = url_bulk + f" {url}"
        
        answer = await agent.run(url_bulk)
        text = answer.output

        results.append({
            "date": date_str,
            "text": text
        })
        
    except FileNotFoundError:
        print(f"File not found: {full_path}")
        missing_dates.append(current_date)
    except pd.errors.EmptyDataError:
        print(f"Empty CSV: {full_path}")
        missing_dates.append(current_date)
    except Exception as e:
        print(f"Error processing {full_path}: {e}")
        missing_dates.append(current_date)
    
    sleep(3)
    current_date += datetime.timedelta(days=1)


df = pd.DataFrame(results)
print(df.head())

path = r"C:\Users\User\dev\training_model\gdelt_data_fetch\gdelt_data1.csv"
df.to_csv(path, index=False)

print(f"============= MISSING DATA FOR TOTAL OF {len(missing_dates)} DATES : =============")
print(missing_dates)


Processing for date: 2014-06-05
Processing for date: 2014-06-06
Processing for date: 2014-06-07
Processing for date: 2014-06-08
Processing for date: 2014-06-09
Processing for date: 2014-06-10
Processing for date: 2014-06-11
Processing for date: 2014-06-12
Processing for date: 2014-06-13
Processing for date: 2014-06-14
Processing for date: 2014-06-15
Processing for date: 2014-06-16
Processing for date: 2014-06-17
Processing for date: 2014-06-18
Processing for date: 2014-06-19
Processing for date: 2014-06-20
Processing for date: 2014-06-21
Processing for date: 2014-06-22
Processing for date: 2014-06-23
Processing for date: 2014-06-24
Processing for date: 2014-06-25
Processing for date: 2014-06-26
Processing for date: 2014-06-27
Processing for date: 2014-06-28
Processing for date: 2014-06-29
Processing for date: 2014-06-30
Processing for date: 2014-07-01
Processing for date: 2014-07-02
Processing for date: 2014-07-03
Processing for date: 2014-07-04
Processing for date: 2014-07-05
Processi

CancelledError: 

In [50]:
import datetime
import os
import pandas as pd
from time import sleep

SYSTEM_PROMPT = """
    You are a helpful assistant that converts the url link of a news article to sentence like topic. As an input you will get bulk of links as string.
    Your task is to return two or maximum three sentences that are going to give some type of description about what happened to gold market or gold prices.
    While generating the answer make sure that:
    -Try to approximate the situation as close as you can.
    -Answer only the topic, do not add extra discussion. 
    -Do not format the answer.  
    Make sure that you capture all the necessary information and the details about the text that is provided you.
"""

provider = GoogleProvider(api_key=API_KEY)
model = GoogleModel("gemini-2.0-flash", provider=provider)
agent = Agent(model=model, system_prompt=SYSTEM_PROMPT, output_type=str, output_retries=3)

base_path = r"C:\Users\User\dev\training_model\gdelt_data_fetch\gdelt_gold_filtered"
start_date = datetime.date(2015, 12, 27)
end_date = datetime.date(2025, 8, 10)

results = []
missing_dates = []
current_date = start_date

while current_date <= end_date:
    # Prepare batch of 2 days
    batch_dates = [current_date]
    next_date = current_date + datetime.timedelta(days=1)
    if next_date <= end_date:
        batch_dates.append(next_date)
    
    print(f"Processing batch for dates: {[d.strftime('%Y-%m-%d') for d in batch_dates]}")

    url_bulk = ""
    missing_in_batch = []

    for date in batch_dates:
        date_str = date.strftime("%Y%m%d")
        file_name = f"{date_str}_gold_filtered.csv"
        full_path = os.path.join(base_path, file_name)

        try:
            df = pd.read_csv(full_path)
            for url in df['SOURCEURL']:
                url_bulk += f" {url}"
        except FileNotFoundError:
            print(f"File not found: {full_path}")
            missing_in_batch.append(date)
        except pd.errors.EmptyDataError:
            print(f"Empty CSV: {full_path}")
            missing_in_batch.append(date)
        except Exception as e:
            print(f"Error processing {full_path}: {e}")
            missing_in_batch.append(date)
    
    if url_bulk.strip():
        answer = await agent.run(url_bulk)
        text = answer.output

        # Store same answer text for each date in batch (or split later if you want)
        for date in batch_dates:
            if date not in missing_in_batch:
                results.append({
                    "date": date.strftime("%Y%m%d"),
                    "text": text
                })
            else:
                missing_dates.append(date)
    else:
        # No data found for both dates
        missing_dates.extend(missing_in_batch)
    
    sleep(3)
    current_date += datetime.timedelta(days=2)  # Move forward by 2 days

df = pd.DataFrame(results)
print(df.head())

path = r"C:\Users\User\dev\training_model\gdelt_data_fetch\gdelt_data2.csv"
df.to_csv(path, index=False)

print(f"============= MISSING DATA FOR TOTAL OF {len(missing_dates)} DATES : =============")
print(missing_dates)


Processing batch for dates: ['2015-12-27', '2015-12-28']
Processing batch for dates: ['2015-12-29', '2015-12-30']
Processing batch for dates: ['2015-12-31', '2016-01-01']
Processing batch for dates: ['2016-01-02', '2016-01-03']
Processing batch for dates: ['2016-01-04', '2016-01-05']
Processing batch for dates: ['2016-01-06', '2016-01-07']
Processing batch for dates: ['2016-01-08', '2016-01-09']
Processing batch for dates: ['2016-01-10', '2016-01-11']
Processing batch for dates: ['2016-01-12', '2016-01-13']
Processing batch for dates: ['2016-01-14', '2016-01-15']
Processing batch for dates: ['2016-01-16', '2016-01-17']
Processing batch for dates: ['2016-01-18', '2016-01-19']


ServerError: 500 INTERNAL. {'error': {'code': 500, 'message': 'Internal error encountered.', 'status': 'INTERNAL'}}

Using **Ollama** local model
 
Because of the limited requests from the online LLMs i will use Ollama model locally.

In [2]:
import os 
import asyncio
import datetime
import pandas as pd 
from time import sleep 
import ollama

In [3]:
SYSTEM_PROMPT = """
You are a helpful assistant that converts the url link of a news article to sentence like topic. As an input you will get bulk of links as string.
Your task is to return two or maximum three sentences that are going to give some type of description about what happened to gold market or gold prices.
While generating the answer make sure that:
-Try to approximate the situation as close as you can.
-Answer only the topic, do not add extra discussion.
-Do not format the answer.
Make sure that you capture all the necessary information and the details about the text that is provided you.
"""

In [5]:
base_path = r"C:\Users\User\dev\training_model\gdelt_data_fetch\gdelt_gold_filtered"
start_date = datetime.date(2015, 12, 27)
end_date = datetime.date(2025, 8, 10)

In [13]:
async def generate_summary(prompt: str) -> str:
    response = await ollama.chat(
        model="gemma:2b",  
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": prompt}
        ]
    )
    return response['message']['content']

In [15]:

results = []
missing_dates = []
current_date = start_date

while current_date <= end_date:
    batch_dates = [current_date]
    next_date = current_date + datetime.timedelta(days=1)
    if next_date <= end_date:
        batch_dates.append(next_date)

    print(f"Processing batch for dates: {[d.strftime('%Y-%m-%d') for d in batch_dates]}")

    url_bulk = ""
    missing_in_batch = []

    for date in batch_dates:
        date_str = date.strftime("%Y%m%d")
        file_name = f"{date_str}_gold_filtered.csv"
        full_path = os.path.join(base_path, file_name)

        try:
            df = pd.read_csv(full_path)
            for url in df['SOURCEURL']:
                url_bulk += f" {url}"
        except FileNotFoundError:
            print(f"File not found: {full_path}")
            missing_in_batch.append(date)
        except pd.errors.EmptyDataError:
            print(f"Empty CSV: {full_path}")
            missing_in_batch.append(date)
        except Exception as e:
            print(f"Error processing {full_path}: {e}")
            missing_in_batch.append(date)

    if url_bulk.strip():
        answer_text = await generate_summary(url_bulk)
        for date in batch_dates:
            if date not in missing_in_batch:
                results.append({
                    "date": date.strftime("%Y%m%d"),
                    "text": answer_text
                })
            else:
                missing_dates.append(date)
    else:
        missing_dates.extend(missing_in_batch)

    sleep(3)
    current_date += datetime.timedelta(days=2)


df = pd.DataFrame(results)
print(df.head())

path = r"C:\Users\User\dev\training_model\gdelt_data_fetch\gdelt_data2.csv"
df.to_csv(path, index=False)

Processing batch for dates: ['2015-12-27', '2015-12-28']


TypeError: object ChatResponse can't be used in 'await' expression

In [16]:
results

[]

In [18]:
#--- works really slow

In [19]:
import datetime
import os
import pandas as pd
from time import sleep
import ollama

SYSTEM_PROMPT = """
You are a helpful assistant that converts the url link of a news article to sentence like topic.
Your task is to return 2-3 sentences describing what happened in the gold market or gold prices.
Answer only the topic, no extra discussion, no formatting.
"""

base_path = r"C:\Users\User\dev\training_model\gdelt_data_fetch\gdelt_gold_filtered"
start_date = datetime.date(2015, 12, 27)
end_date = datetime.date(2025, 8, 10)
MODEL_NAME = "gemma:2b"  # small model for low RAM

def generate_summary(prompt: str) -> str:
    response = ollama.chat(
        model=MODEL_NAME,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": prompt}
        ]
    )
    return response['message']['content']

results = []
missing_dates = []
current_date = start_date

while current_date <= end_date:
    batch_dates = [current_date]
    next_date = current_date + datetime.timedelta(days=1)
    if next_date <= end_date:
        batch_dates.append(next_date)

    print(f"Processing batch for dates: {[d.strftime('%Y-%m-%d') for d in batch_dates]}")

    url_bulk = []
    missing_in_batch = []

    for date in batch_dates:
        date_str = date.strftime("%Y%m%d")
        file_name = f"{date_str}_gold_filtered.csv"
        full_path = os.path.join(base_path, file_name)

        try:
            df = pd.read_csv(full_path)
            url_bulk.extend(df['SOURCEURL'].tolist())
        except FileNotFoundError:
            print(f"File not found: {full_path}")
            missing_in_batch.append(date)
        except pd.errors.EmptyDataError:
            print(f"Empty CSV: {full_path}")
            missing_in_batch.append(date)
        except Exception as e:
            print(f"Error processing {full_path}: {e}")
            missing_in_batch.append(date)

    if url_bulk:
        # Split into chunks of 20 URLs to make model faster
        chunk_size = 20
        summaries = []
        for i in range(0, len(url_bulk), chunk_size):
            chunk = " ".join(url_bulk[i:i+chunk_size])
            summary = generate_summary(chunk)
            summaries.append(summary)

        # Combine summaries for the batch
        final_text = " ".join(summaries)

        for date in batch_dates:
            if date not in missing_in_batch:
                results.append({
                    "date": date.strftime("%Y%m%d"),
                    "text": final_text
                })
            else:
                missing_dates.append(date)
    else:
        missing_dates.extend(missing_in_batch)

    sleep(1)
    current_date += datetime.timedelta(days=2)

df = pd.DataFrame(results)
path = r"C:\Users\User\dev\training_model\gdelt_data_fetch\gdelt_data2.csv"
df.to_csv(path, index=False)

print(f"Processed {len(results)} entries. Missing dates: {len(missing_dates)}")


Processing batch for dates: ['2015-12-27', '2015-12-28']
Processing batch for dates: ['2015-12-29', '2015-12-30']


KeyboardInterrupt: 

In [20]:
results

[{'date': '20151227',
  'text': 'I am unable to generate the requested information as I am unable to access external sources or provide real-time news updates. I am unable to generate a sentence about the topic from the context, as the context does not provide any information about the gold market or gold prices. I am unable to generate the requested 2-3 sentences as I am unable to access external news sources or provide commentary on current events. I am unable to generate the requested information from the context, as I am unable to access external sources and provide relevant data. I am unable to generate the requested information due to the lack of context and data in the context. The articles provide no information about the current state of the gold market or prices. Therefore, I cannot generate the requested topic. I am unable to provide a 2-3 sentence summary of the gold market or gold prices, as I do not have access to real-time or financial news sources. I am unable to genera

In [3]:
import datetime
import os
import pandas as pd
from dotenv import load_dotenv
from time import sleep
from pydantic_ai import Agent
from pydantic_ai.providers.openai import OpenAIProvider
from pydantic_ai.models.openai import OpenAIModel

SYSTEM_PROMPT = """
    You are a helpful assistant that converts the url link of a news article to sentence like topic. As an input you will get bulk of links as string.
    Your task is to return two or maximum three sentences that are going to give some type of description about what happened to gold market or gold prices.
    While generating the answer make sure that:
    -Try to approximate the situation as close as you can.
    -Answer only the topic, do not add extra discussion. 
    -Do not format the answer.  
    Make sure that you capture all the necessary information and the details about the text that is provided you.
"""

load_dotenv(override=True)
API_KEY = os.environ['OPENAI2']

provider = OpenAIProvider(api_key=API_KEY)
model = OpenAIModel("gpt-4o-mini", provider=provider)

# Create the agent
agent = Agent(
    model=model,
    system_prompt=SYSTEM_PROMPT,
    output_type=str,
    output_retries=3
)

base_path = r"C:\Users\User\dev\training_model\gdelt_data_fetch\gdelt_gold_filtered"
start_date = datetime.date(2015, 12, 27)
end_date = datetime.date(2025, 8, 10)

results = []
missing_dates = []
current_date = start_date

while current_date <= end_date:
    # Prepare batch of 2 days
    batch_dates = [current_date]
    next_date = current_date + datetime.timedelta(days=1)
    if next_date <= end_date:
        batch_dates.append(next_date)
    
    print(f"Processing batch for dates: {[d.strftime('%Y-%m-%d') for d in batch_dates]}")

    url_bulk = ""
    missing_in_batch = []

    for date in batch_dates:
        date_str = date.strftime("%Y%m%d")
        file_name = f"{date_str}_gold_filtered.csv"
        full_path = os.path.join(base_path, file_name)

        try:
            df = pd.read_csv(full_path)
            for url in df['SOURCEURL']:
                url_bulk += f" {url}"
        except FileNotFoundError:
            print(f"File not found: {full_path}")
            missing_in_batch.append(date)
        except pd.errors.EmptyDataError:
            print(f"Empty CSV: {full_path}")
            missing_in_batch.append(date)
        except Exception as e:
            print(f"Error processing {full_path}: {e}")
            missing_in_batch.append(date)
    
    if url_bulk.strip():
        answer = await agent.run(url_bulk)
        text = answer.output

        # Store same answer text for each date in batch (or split later if you want)
        for date in batch_dates:
            if date not in missing_in_batch:
                results.append({
                    "date": date.strftime("%Y%m%d"),
                    "text": text
                })
            else:
                missing_dates.append(date)
    else:
        # No data found for both dates
        missing_dates.extend(missing_in_batch)
    
    sleep(3)
    current_date += datetime.timedelta(days=2)  # Move forward by 2 days

df = pd.DataFrame(results)
print(df.head())

path = r"C:\Users\User\dev\training_model\gdelt_data_fetch\gdelt_data2.csv"
df.to_csv(path, index=False)

print(f"============= MISSING DATA FOR TOTAL OF {len(missing_dates)} DATES : =============")
print(missing_dates)


Processing batch for dates: ['2015-12-27', '2015-12-28']


ModelHTTPError: status_code: 429, model_name: gpt-4o-mini, body: {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}